# 01 — Data Quality Gate

This notebook queries DuckDB and never loads the full 3.5 GB inventory into pandas.


In [ ]:
from pathlib import Path
import duckdb

DB = Path.home() / "Documents" / "crawler_result" / "STORAGE_ANALYSIS.duckdb"
if not DB.exists():
    raise FileNotFoundError(f"DuckDB database not found: {DB}")

con = duckdb.connect(str(DB), read_only=True)
print(DB)


In [ ]:
quality = con.sql("SELECT * FROM data_quality").df()
quality


In [ ]:
con.sql("""
SELECT
    FileCategory,
    COUNT(*) AS file_count,
    SUM(SizeGB) AS storage_gb
FROM storage_detail
WHERE ItemType = 'File'
GROUP BY FileCategory
ORDER BY storage_gb DESC
""").df()


In [ ]:
# Duplicate-path investigation; the result stays small.
con.sql("""
SELECT FullPath, COUNT(*) AS occurrences
FROM storage_detail
GROUP BY FullPath
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
LIMIT 100
""").df()
